# Lesson 11: Prompt engineering

*Companion notebook for [the lesson page](../lessons/11-prompt-engineering.html). Worked solutions are in `solutions/11-prompt-engineering-solutions.ipynb`; try each exercise first.*

Write prompts that work reliably, not just once, through clear instructions, context, examples and structure, along with reasoning and chaining techniques, and treat prompts as tested, versioned code.

**Runs on:** Any computer, plus an API key.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q anthropic python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()   # reads ANTHROPIC_API_KEY from a .env file next to this notebook, if there is one
# On Google Colab, add the key under the key icon (Secrets), then uncomment the next two lines:
# from google.colab import userdata
# os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"          # check your provider's current model list
print("API key found" if os.environ.get("ANTHROPIC_API_KEY") else
      "No API key set: cells marked 'Needs an API key' will fail until you add one")

In [ ]:
def llm(prompt: str, max_tokens: int = 1000) -> str:
    """One-shot helper used by the chaining examples."""
    r = client.messages.create(model=MODEL, max_tokens=max_tokens,
                               messages=[{"role": "user", "content": prompt}])
    return r.content[0].text

## Principles

### 1. Be explicit about the task, the audience and the output

Compare:

*Vague*

```text
Summarise this incident report.
```

*Specific*

```text
Summarise the incident report below for the on-call engineering manager, who will
decide within five minutes whether to wake up the hardware team.

Write at most 5 bullet points:
- what failed, where, and since when
- current impact on data collection
- what has already been tried
- the single most likely cause, if the report supports one
- the decision needed, phrased as a yes/no question

If the report does not say something, write "not stated" rather than guessing.
```

### 4. Structure the prompt

Long prompts mix instructions, reference material and input data. Separate them clearly so the model never confuses data with instructions. XML-style tags work well and are easy to parse back out:

```text
<instructions>
Classify the support email into exactly one category and explain briefly.
</instructions>

<categories>
hardware: physical faults, power, cables, sensors, enclosures
software: firmware, drivers, the acquisition application, updates
data: missing, corrupted or suspicious measurements, processing results
other: anything else, including billing and scheduling
</categories>

<email>
{email_text}
</email>

Answer in the form:
<category>...</category>
<reason>one sentence</reason>
```

## Techniques

### Few-shot prompting

Include worked input-output pairs. In a chat API you can write them as earlier conversation turns, which models treat as strong evidence of the desired behaviour.

In [ ]:
new_reading = "O3 reads exactly 0.0 ppb for ten minutes every day at 12:00"
messages = [
    {"role": "user", "content": "Reading: 'NO2 412 ppb at 03:00, all other hours 10-25 ppb'"},
    {"role": "assistant", "content": "SUSPECT: isolated spike roughly 20x the surrounding values."},
    {"role": "user", "content": "Reading: 'NO2 rises from 15 to 38 ppb between 07:00 and 09:00'"},
    {"role": "assistant", "content": "PLAUSIBLE: gradual morning increase consistent with traffic."},
    {"role": "user", "content": f"Reading: '{new_reading}'"},
]

In [ ]:
# Needs an API key (see the setup cell)
r = client.messages.create(model=MODEL, max_tokens=100, messages=messages)
print(r.content[0].text)

### Let the model reason before answering

For multi-step problems, asking the model to work through the problem before giving the final answer improves accuracy, because each generated token is extra computation the model can build on. This is **chain-of-thought** prompting. Ask for the reasoning in a separate section so your code can discard it and keep only the answer:

```text
Work out the answer step by step inside <thinking> tags.
Then give only the final number inside <answer> tags.
```

### Prompt chaining

One giant prompt that researches, drafts, critiques and formats is hard to debug. Break the job into steps, each a separate call with a focused prompt, passing outputs forward:

In [ ]:
def answer_with_review(question: str, document: str) -> str:
    quotes = llm(f"<doc>{document}</doc>\nList verbatim quotes relevant to: {question}")
    draft = llm(f"<quotes>{quotes}</quotes>\nAnswer '{question}' using only these quotes, citing them.")
    review = llm(f"<quotes>{quotes}</quotes>\n<answer>{draft}</answer>\n"
                 "List any claim in the answer not supported by the quotes, or reply NONE.")
    if review.strip() != "NONE":
        draft = llm(f"Rewrite the answer removing unsupported claims.\n<problems>{review}</problems>\n<answer>{draft}</answer>")
    return draft

In [ ]:
# Needs an API key (see the setup cell)
doc = ("Replace the desiccant cartridge every six months, or every three months at humid sites. "
       "The indicator window turns pink when the desiccant is saturated.")
print(answer_with_review("How often should the desiccant be replaced at a humid site?", doc))

## Prompts as code

### Templates

Keep prompts in files or constants, separate from logic, and fill them with variables. Escape or delimit user input so it cannot break your structure.

In [ ]:
from string import Template

CLASSIFY_V3 = Template("""<instructions>
Classify the email into exactly one category: hardware, software, data or other.
</instructions>
<email>
$email
</email>
Reply with only the category name, in lower case.""")

def classify(email: str) -> str:
    prompt = CLASSIFY_V3.substitute(email=email.replace("</email>", ""))
    r = client.messages.create(model=MODEL, max_tokens=5, temperature=0,
                               messages=[{"role": "user", "content": prompt}])
    return r.content[0].text.strip().lower()

### Test prompts against examples, not vibes

A prompt that looks great on three examples you tried can fail on the fourth. Build a small labelled set (even 20 to 50 cases, including tricky ones), run every prompt version against it, and track the score. This is the seed of the evaluation discipline in lesson 17, and the single habit that most separates professionals from tinkerers.

In [ ]:
cases = [
    ("The fan in the enclosure is grinding loudly", "hardware"),
    ("After updating to 2.4 the GUI crashes on start", "software"),
    ("Yesterday's ozone column values are all exactly zero", "data"),
    ("Can we move the calibration visit to March?", "other"),
    # ... at least 20, including ambiguous ones
]

def score(classify_fn):
    results = [(text, gold, classify_fn(text)) for text, gold in cases]
    acc = sum(g == p for _, g, p in results) / len(results)
    for text, gold, pred in results:
        if gold != pred:
            print(f"MISS  gold={gold:<9} pred={pred:<9} {text}")
    return acc

In [ ]:
# Needs an API key (see the setup cell)
print("accuracy:", score(classify))

### Hands-on: iterate a classifier prompt to 90%+

1. Write 30 short support messages across the four categories above. Make at least 8 deliberately ambiguous ones (a firmware update that bricked a sensor; a data gap caused by a power cut).
2. Write the simplest possible classification prompt and score it with `score()`.
3. Iterate: add category definitions, add tie-break rules for the ambiguous cases ("classify by root cause if stated, otherwise by symptom"), add two or three examples. Score each version and keep a table in your log.
4. Try a smaller, cheaper model with your best prompt. Is the accuracy loss acceptable for the cost saving?

In [ ]:
# Your code here

## Check yourself

Take the quiz at the end of [the lesson page](../lessons/11-prompt-engineering.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.